# Marine Debris 3-Model Comparison — RunPod Notebook

This notebook runs the packaged experiment script for YOLOv8s, Faster R-CNN, and MobileNet SSD. Edit the dataset paths, run the quick debug, then run the full experiment.

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

## 1) Create config.yaml
Edit the paths below to match your RunPod dataset locations.

In [ ]:
from pathlib import Path
config = r'''
trash_root: /workspace/datasets/trash-icra19
river_root: /workspace/datasets/river-floating-trash
out_dir: /workspace/runs/marine_3model_comparison

class_names: [plastic, bio, rov]
seed: 42
split_mode: stratified_70_15_15
copy_files: false

training:
  epochs_head: 10
  epochs_finetune: 100
  imgsz_yolo: 640
  imgsz_frcnn: 640
  imgsz_ssd: 320
  batch_yolo: 16
  batch_frcnn: 8
  batch_ssd: 16
  lr_yolo: 0.01
  lr_torch: 0.001
  workers: 8
  amp: true

thresholds:
  conf_yolo: 0.25
  conf_torch: 0.50
  iou_match: 0.50

run:
  prepare_data: true
  train_yolo: true
  train_frcnn: true
  train_ssd: true
  evaluate: true
  quick_debug: false
'''
Path('config.yaml').write_text(config, encoding='utf-8')
print(Path('config.yaml').read_text())

## 2) Quick debug run
This checks paths, labels, imports, and the training pipeline with a tiny 1-epoch run.

In [ ]:
!python marine_3model_experiment.py --config config.yaml --quick-debug

## 3) Full run
Run this only after the quick debug works.

In [ ]:
!python marine_3model_experiment.py --config config.yaml

## 4) Display paper-ready tables

In [ ]:
import pandas as pd
out_dir = '/workspace/runs/marine_3model_comparison'
for name in ['dataset_summary.csv', 'results_overall_test.csv', 'results_per_class_map.csv', 'results_cross_domain.csv']:
    p = Path(out_dir) / name
    print('\n###', name)
    if p.exists():
        display(pd.read_csv(p))
    else:
        print('Missing:', p)